# <font color="#418FDE" size="6.5" uppercase>**Text als Merkmale**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Strukturieren und bereinigen kleine lokale Textsammlungen mit Labels. 
- Erzeugen Textmerkmale mit Count-, TF-IDF- und Hashing-Vektorisierung. 
- Trainieren, validieren, analysieren und speichern einfache Textklassifikationspipelines. 


## **1. Texte vorbereiten**

### **1.1. Lokale Textsammlung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_01.jpg?v=1787681551" width="250">



>* Lokale Textsammlungen sind stabil und überprüfbar
>* Jeder Text braucht ein klares Label

>* Texte klar ordnen und Analyseeinheiten festlegen
>* Labels vereinheitlichen, Duplikate und Herkunft prüfen

>* Daten auf Verzerrungen und Grenzen prüfen
>* Einzelbeispiele lesen und Labels besser verstehen



In [ ]:
#@title Python-Code - Lokale Textsammlung

# Diese Sammlung zeigt lokale Texte mit Labels.
# Wir prüfen Ordnung, Duplikate und Labelkonsistenz.
# Am Ende entsteht eine bereinigte Übersicht.

import pandas as pd

# Kleine Texte stehen hier direkt im Notebook.
raw_data = [
    {"text": "Die Rechnung ist falsch berechnet.", "label": "Rechnung"},
    {"text": "Mein Login funktioniert seit gestern nicht.", "label": "Technik"},
    {"text": "Ich möchte meinen Vertrag kündigen.", "label": "Kündigung"},
]

# Einige Einträge enthalten typische Qualitätsprobleme.
raw_data.extend(
    [
        {"text": "Die Rechnung ist falsch berechnet.", "label": "Reklamation"},
        {"text": "  Bitte senden Sie mir die Rechnung erneut. ", "label": "rechnung"},
    ]
)

# Aus der Liste entsteht eine kleine lokale Tabelle.
texts = pd.DataFrame(raw_data)

# Text und Label werden einheitlich bereinigt.
texts["clean_text"] = texts["text"].str.strip()
texts["clean_label"] = texts["label"].str.strip().str.lower()

# Unterschiedliche Schreibweisen werden auf feste Labels abgebildet.
label_map = {
    "rechnung": "Rechnung",
    "reklamation": "Rechnung",
    "technik": "Technik",
    "kündigung": "Kündigung",
}
texts["final_label"] = texts["clean_label"].map(label_map)

# Diese Prüfung schützt vor unbekannten Labels.
missing_labels = texts["final_label"].isna().sum()
if missing_labels != 0:
    raise ValueError("Mindestens ein Label ist nicht im Labelplan enthalten.")

# Doppelte Texte werden für die Modellvorbereitung entfernt.
clean_collection = texts.drop_duplicates(subset="clean_text", keep="first")

# Eine kompakte Übersicht zeigt die bereinigte Sammlung.
summary = clean_collection[["clean_text", "final_label"]]
print("Bereinigte lokale Textsammlung:")
print(summary.to_string(index=False))

# Die Labelverteilung zeigt mögliche Schieflagen frühzeitig.
label_counts = clean_collection["final_label"].value_counts().sort_index()
print("Labelverteilung:", label_counts.to_dict())
print("Entfernte Duplikate:", len(texts) - len(clean_collection))



### **1.2. Texte bereinigen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_02.jpg?v=1787681554" width="250">



>* Texte vergleichbar und zuverlässig vorbereiten
>* Relevante Inhalte von technischem Rauschen trennen

>* Texte vereinheitlichen, Bedeutung erhalten
>* Kontext entscheidet, was wichtig bleibt

>* Bereinigung einheitlich und nachvollziehbar durchführen
>* Duplikate, leere Texte und Labels prüfen



In [ ]:
#@title Python-Code - Texte bereinigen

# Wir bereinigen kurze Texte für spätere Merkmale.
# Der Fokus liegt auf konsistenter Vorverarbeitung.
# Am Ende vergleichen wir Rohtext und Bereinigung.

import pandas as pd

# Diese kleine Sammlung enthält typische Störungen.
raw_data = pd.DataFrame(
    {
        "label": ["positiv", "negativ", "positiv", "negativ"],
        "text": [
            "  TOLLES Produkt!!! <br> Lieferung war schnell.  ",
            "Negativ: Akku leer... Modell X-200 Fehlercode 17.",
            "Tolles Produkt!!! Lieferung war schnell.",
            "   ",
        ],
    }
)

# Eine einfache Funktion macht die Regeln nachvollziehbar.
def clean_text(text):
    cleaned = text.lower()
    cleaned = cleaned.replace("<br>", " ")
    cleaned = cleaned.replace("negativ:", " ")
    cleaned = cleaned.replace("!", " ")
    cleaned = cleaned.replace(".", " ")
    cleaned = " ".join(cleaned.split())
    return cleaned

# Die Bereinigung wird auf alle Dokumente gleich angewendet.
cleaned_data = raw_data.copy()
cleaned_data["clean_text"] = cleaned_data["text"].apply(clean_text)

# Leere Dokumente werden sichtbar markiert und entfernt.
cleaned_data["is_empty"] = cleaned_data["clean_text"].str.len() == 0
usable_data = cleaned_data.loc[~cleaned_data["is_empty"]].copy()

# Duplikate nach der Bereinigung werden ebenfalls geprüft.
duplicate_count = usable_data["clean_text"].duplicated().sum()
usable_data = usable_data.drop_duplicates(subset="clean_text")

# Eine kleine Prüfung schützt vor unbemerkten Datenproblemen.
if len(usable_data) == 0:
    raise ValueError("Nach der Bereinigung bleibt kein nutzbarer Text übrig.")

# Die Ausgabe zeigt nur die wichtigsten Qualitätskontrollen.
print("Bereinigte Textsammlung")
print("Dokumente vorher:", len(raw_data))
print("Leere Dokumente entfernt:", int(cleaned_data["is_empty"].sum()))
print("Duplikate entfernt:", int(duplicate_count))
print("Dokumente nachher:", len(usable_data))
print(usable_data[["label", "clean_text"]].to_string(index=False))



### **1.3. Wörter zählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_03.jpg?v=1787681552" width="250">



>* Wortzahlen machen Texte vergleichbar und auswertbar
>* Häufigkeiten zeigen Themen und mögliche Labelmuster

>* Wortdefinitionen beeinflussen spätere Textanalysen.
>* Zählregeln müssen zur Aufgabe passen.

>* Häufige Wörter zeigen Muster und Verzerrungen.
>* Wortzählung prüft Datenqualität und Modellgrundlagen.



In [ ]:
#@title Python-Code - Wörter zählen

# Dieses Beispiel zählt Wörter in kurzen Texten.
# Es zeigt Bereinigung vor der Zählung.
# Die Ausgabe vergleicht Wörter nach Labels.

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

# Eine kleine lokale Textsammlung steht direkt im Code.
texts = [
    "Lieferung war spät, Paket beschädigt.",
    "Der Service war langsam und die Lieferung spät.",
    "Schnelle Lieferung, tolles Paket, sehr zufrieden!",
    "Ich bin zufrieden und empfehle den schnellen Service.",
]

# Jedes Dokument bekommt ein einfaches Label.
labels = ["Beschwerde", "Beschwerde", "Lob", "Lob"]

# Die Längenprüfung verhindert unpassende Text-Label-Paare.
if len(texts) != len(labels):
    raise ValueError("Jeder Text braucht genau ein Label.")

# CountVectorizer bereinigt grob und zählt Wörter automatisch.
vectorizer = CountVectorizer(lowercase=True)
word_matrix = vectorizer.fit_transform(texts)

# Die Wortliste bestimmt die Spalten der Zählmatrix.
words = vectorizer.get_feature_names_out()
counts = word_matrix.toarray()

# Eine Tabelle macht Dokumente, Labels und Wortzahlen sichtbar.
count_table = pd.DataFrame(counts, columns=words)
count_table.insert(0, "label", labels)

# Pro Label werden die Wortzahlen zusammengezählt.
label_counts = count_table.groupby("label").sum()

# Wir wählen die häufigsten Wörter insgesamt aus.
total_counts = label_counts.sum(axis=0)
top_words = total_counts.sort_values(ascending=False).head(5).index

# Die Ausgabe bleibt kurz und zeigt die wichtigsten Ergebnisse.
print("Dokumente:", len(texts))
print("Unterschiedliche Wörter:", len(words))
print("Häufigste Wörter:", ", ".join(top_words))
print("Beschwerde zählt 'spät':", int(label_counts.loc["Beschwerde", "spät"]))
print("Lob zählt 'zufrieden':", int(label_counts.loc["Lob", "zufrieden"]))

# Das Diagramm vergleicht die Top-Wörter nach Label.
plot_data = label_counts.loc[:, top_words].T
ax = plot_data.plot(kind="bar", figsize=(8, 4))

# Achsentitel erklären, was gezählt wurde.
ax.set_title("Wortzählungen nach Label")
ax.set_xlabel("Wort")
ax.set_ylabel("Anzahl")

# Die Legende hilft beim Vergleich der Labels.
ax.legend(title="Label")
plt.tight_layout()
plt.show()



## **2. Texte vektorisieren**

### **2.1. n Gramme und Stopwörter**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_01.jpg?v=1787681543" width="250">



>* n-Gramme erfassen kurze Wortfolgen als Merkmale
>* Sie bewahren Kontext und typische Formulierungen

>* Unigramme sind robust, Bigramme oft präziser
>* Vektorisierung bestimmt nutzbare Sprachmuster

>* Stopwörter können Merkmale kompakter machen
>* Entfernung sorgfältig nach Kontext entscheiden



In [ ]:
#@title Python-Code - n Gramme und Stopwörter

# Dieses Beispiel zeigt n-Gramme und Stopwörter.
# Wir vergleichen Merkmale aus kurzen deutschen Texten.
# Die Ausgabe macht entfernte Wörter sichtbar.

from sklearn.feature_extraction.text import CountVectorizer
import sklearn

# Kleine Texte halten das Beispiel übersichtlich.
texts = [
    "Das Produkt ist nicht gut",
    "Das Produkt ist sehr gut",
    "Der Service ist nicht schnell",
    "Der Service ist sehr schnell",
]

# Diese Stopwortliste entfernt häufige Wörter, aber nicht Verneinungen.
stop_words = ["das", "der", "ist", "sehr"]

# Unigramme und Bigramme zeigen Wörter und kurze Wortfolgen.
vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    stop_words=stop_words,
)

# Die Matrix enthält Zählwerte für jedes gefundene Merkmal.
feature_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

# Eine einfache Prüfung schützt vor unerwartet leeren Merkmalen.
if feature_matrix.shape[1] == 0:
    raise ValueError("Keine Merkmale gefunden.")

# Wir zeigen nur wenige Merkmale, damit alles lesbar bleibt.
selected_features = list(feature_names[:10])
first_row = feature_matrix.toarray()[0, :10].tolist()

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Texte: {feature_matrix.shape[0]}, Merkmale: {feature_matrix.shape[1]}")
print(f"Stopwörter: {', '.join(stop_words)}")
print(f"Erste Merkmale: {selected_features}")
print(f"Zählwerte im ersten Text: {first_row}")
print("Wichtig: 'nicht gut' bleibt als Bigramm erhalten.")



### **2.2. TF IDF Gewichtung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_02.jpg?v=1787681539" width="250">



>* TF IDF gewichtet Wörter aussagekräftiger
>* Spezifische Begriffe werden stärker hervorgehoben

>* TF IDF reduziert Längen- und Häufigkeitsverzerrungen
>* Seltene, typische Begriffe erhalten mehr Gewicht

>* Einfach, effizient und gut interpretierbar
>* Nicht semantisch; Vorverarbeitung bleibt entscheidend



In [ ]:
#@title Python-Code - TF IDF Gewichtung

# Dieses Beispiel zeigt TF-IDF-Gewichte für kurze Texte.
# Häufige Wörter werden gegenüber seltenen Begriffen abgeschwächt.
# Die Ausgabe vergleicht Count- und TF-IDF-Merkmale.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import sklearn

# Eine kleine Textsammlung bleibt vollständig im Arbeitsspeicher.
documents = [
    "akku akku produkt gut",
    "akku ladezeit produkt gut",
    "lieferung verspätet produkt gut",
    "kundenservice antwortet langsam produkt gut",
]

# CountVectorizer zählt nur, wie oft Wörter vorkommen.
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(documents)

# TfidfVectorizer gewichtet Wörter nach Häufigkeit und Seltenheit.
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Beide Vektorisierer müssen dieselben vier Dokumente enthalten.
if count_matrix.shape[0] != len(documents):
    raise ValueError("Die Anzahl der Dokumente passt nicht.")

# Wir betrachten ausgewählte Wörter aus dem ersten Dokument.
selected_terms = ["akku", "produkt", "gut"]
count_names = list(count_vectorizer.get_feature_names_out())
tfidf_names = list(tfidf_vectorizer.get_feature_names_out())

# Die Werte werden für Anfänger übersichtlich gerundet.
rows = []
for term in selected_terms:
    count_value = count_matrix[0, count_names.index(term)]
    tfidf_value = tfidf_matrix[0, tfidf_names.index(term)]
    rows.append([term, int(count_value), round(float(tfidf_value), 3)])

# Eine kleine Tabelle macht den Unterschied direkt sichtbar.
result = pd.DataFrame(rows, columns=["Wort", "Count", "TF-IDF"])
print("scikit-learn Version:", sklearn.__version__)
print("Dokument 1:", documents[0])
print(result.to_string(index=False))



### **2.3. Hashing und Sparse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_03.jpg?v=1787681541" width="250">



>* Text erzeugt schnell sehr viele Merkmale
>* Hashing ordnet Tokens ohne Wörterbuch Spalten zu

>* Effizient ohne gespeichertes Vokabular
>* Kollisionen erschweren die Interpretation

>* Sparse-Matrizen speichern nur vorhandene Textmerkmale
>* Das spart Speicher und beschleunigt Modelle



In [ ]:
#@title Python-Code - Hashing und Sparse

# Dieses Beispiel zeigt Hashing-Vektoren für kurze Texte.
# Sparse-Matrizen speichern nur vorhandene Textmerkmale effizient.
# Die Ausgabe vergleicht Größe, Dichte und Kollisionen.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
import sklearn

# Eine kleine Textsammlung reicht für das Grundprinzip.
texts = [
    "lieferung schnell paket angekommen",
    "lieferung verspätet paket problem",
    "rechnung zahlung frist kundennummer",
    "support problem fehler kundennummer",
]

# CountVectorizer lernt ein sichtbares Vokabular aus den Texten.
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(texts)

# HashingVectorizer nutzt feste Spalten ohne gespeichertes Vokabular.
hash_vectorizer = HashingVectorizer(
    n_features=8,
    alternate_sign=False,
    norm=None,
)

hash_matrix = hash_vectorizer.transform(texts)

# Diese Prüfung macht die erwartete Sparse-Form explizit.
if count_matrix.shape[0] != len(texts):
    raise ValueError("Die Anzahl der Dokumente passt nicht.")

# Dichte bedeutet Anteil der Nicht-Null-Werte in der Matrix.
count_density = count_matrix.nnz / np.prod(count_matrix.shape)
hash_density = hash_matrix.nnz / np.prod(hash_matrix.shape)

# Kollisionen werden hier durch wenige Hash-Spalten sichtbar.
unique_tokens = count_matrix.shape[1]
used_hash_columns = len(set(hash_matrix.nonzero()[1]))
possible_collisions = unique_tokens - used_hash_columns

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Count-Form: {count_matrix.shape}, Dichte: {count_density:.2f}")
print(f"Hashing-Form: {hash_matrix.shape}, Dichte: {hash_density:.2f}")
print(f"Gelernte Count-Wörter: {unique_tokens}")
print(f"Benutzte Hash-Spalten: {used_hash_columns}")
print(f"Mögliche Kollisionen: {possible_collisions}")

# Die Grafik zeigt, wie dünn die Hashing-Matrix besetzt ist.
fig, ax = plt.subplots(figsize=(7, 3))
ax.spy(hash_matrix, markersize=12)
ax.set_title("Sparse-Struktur einer Hashing-Textmatrix")
ax.set_xlabel("Hash-Spalte")
ax.set_ylabel("Dokument")
plt.show()



## **3. Textpipeline trainieren**

### **3.1. Naive Bayes Klassifikation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_01.jpg?v=1787681547" width="250">



>* Lernt typische Wörter aus gelabelten Texten
>* Schnell, einfach, trotz naiver Unabhängigkeitsannahme

>* Vektorisierte Texte liefern Naive Bayes Merkmale
>* Datenvorbereitung beeinflusst die Klassifikationsqualität

>* Fehlerarten im Anwendungskontext gezielt prüfen
>* Validierte Pipeline speichern und reproduzierbar nutzen



In [ ]:
#@title Python-Code - Naive Bayes Klassifikation

# Wir trainieren eine kleine Textpipeline.
# Naive Bayes klassifiziert vektorisierte Texte.
# Die Ausgabe zeigt Güte und Fehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Diese Mini-Daten simulieren gelabelte Kundenrückmeldungen.
texts = [
    "defekt rückerstattung wartezeit ärgerlich",
    "gerät kaputt bitte ersatz sofort",
    "lange wartezeit hotline problem",
    "rechnung falsch beschwerde service",
    "danke hilfreich freundlich zufrieden",
    "super service danke schnell",
    "zufrieden mit beratung und hilfe",
    "freundlich kompetent danke team",
    "wie ändere ich meine adresse",
    "frage zu lieferung und termin",
    "wann kommt meine bestellung",
    "hilfe bitte passwort zurücksetzen",
]

# Die Labels sind die Zielklassen der Klassifikation.
labels = np.array([
    "Beschwerde", "Beschwerde", "Beschwerde", "Beschwerde",
    "Lob", "Lob", "Lob", "Lob",
    "Frage", "Frage", "Frage", "Frage",
])

# Diese Prüfung macht die Datenannahme sichtbar.
if len(texts) != len(labels):
    raise ValueError("Jeder Text braucht genau ein Label.")

# Die Aufteilung trennt Training und Validierung sauber.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.33, random_state=42, stratify=labels
)

# CountVectorizer erzeugt Wortzählungen nur aus Trainingsdaten.
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(lowercase=True)),
    ("model", MultinomialNB()),
])

# Die Pipeline lernt Vokabular und Klassenmuster gemeinsam.
pipeline.fit(train_texts, train_labels)

# Neue Texte werden mit denselben Schritten vorhergesagt.
predicted_labels = pipeline.predict(test_texts)
accuracy = accuracy_score(test_labels, predicted_labels)

# Kurze Ausgaben fassen das Experiment zusammen.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Validierungsgenauigkeit: {accuracy:.2f}")
print(f"Trainingsbeispiele: {len(train_texts)}, Testbeispiele: {len(test_texts)}")

# Ein einzelnes Beispiel zeigt die praktische Anwendung.
new_text = ["danke für die schnelle hilfe"]
new_prediction = pipeline.predict(new_text)[0]
print(f"Neuer Text wird klassifiziert als: {new_prediction}")

# Die Konfusionsmatrix zeigt richtige und falsche Zuordnungen.
class_names = pipeline.named_steps["model"].classes_
fig, ax = plt.subplots(figsize=(5, 4))

# Die Grafik macht Fehler pro Klasse sichtbar.
ConfusionMatrixDisplay.from_predictions(
    test_labels, predicted_labels, labels=class_names, ax=ax, colorbar=False
)

ax.set_title("Naive Bayes: Validierung der Textpipeline")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
plt.tight_layout()
plt.show()



### **3.2. Modelle vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_02.jpg?v=1787681545" width="250">



>* Modelle systematisch und kontrolliert vergleichen
>* Nur einzelne Pipeline-Schritte gezielt verändern

>* Sauber trennen: Training, Test, Kreuzvalidierung
>* Passende Kennzahlen und wichtige Fehler prüfen

>* Nicht nur Genauigkeit zählt beim Modellvergleich
>* Wähle eine passende, wiederverwendbare Pipeline



In [ ]:
#@title Python-Code - Modelle vergleichen

# Wir vergleichen einfache Textpipelines systematisch.
# Gleiche Merkmale machen Modelle fair vergleichbar.
# Die Grafik zeigt Testgenauigkeit pro Modell.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt

# Diese kleine Textsammlung bleibt vollständig im Arbeitsspeicher.
texts = [
    "Das Paket kam schnell und der Service war freundlich",
    "Die Lieferung war pünktlich und alles funktionierte gut",
    "Ich bin zufrieden mit Qualität und Support",
    "Sehr gute Beratung und schnelle Antwort vom Team",
    "Das Produkt erfüllt meine Erwartungen vollständig",
    "Freundlicher Kontakt und einfache Bestellung",
    "Die App ist übersichtlich und läuft stabil",
    "Gute Verpackung und klare Anleitung im Karton",
    "Der Akku hält lange und das Gerät wirkt hochwertig",
    "Schnelle Hilfe löste mein Problem sofort",
    "Das Paket kam beschädigt und viel zu spät",
    "Der Support antwortete nicht auf meine Beschwerde",
    "Ich bin enttäuscht von Qualität und Preis",
    "Die App stürzt ständig ab und ist langsam",
    "Das Produkt funktioniert nicht wie beschrieben",
    "Unfreundlicher Kontakt und komplizierte Rückgabe",
    "Die Anleitung fehlt und die Verpackung war offen",
    "Der Akku ist schwach und das Gerät wird heiß",
    "Meine Bestellung wurde falsch geliefert",
    "Keine Hilfe trotz mehrfacher Nachfrage",
]

# Die Labels beschreiben zwei Klassen der kurzen Texte.
labels = [
    "positiv", "positiv", "positiv", "positiv", "positiv",
    "positiv", "positiv", "positiv", "positiv", "positiv",
    "negativ", "negativ", "negativ", "negativ", "negativ",
    "negativ", "negativ", "negativ", "negativ", "negativ",
]

# Eine einfache Prüfung verhindert unbemerkte Datenfehler.
if len(texts) != len(labels):
    raise ValueError("Jeder Text braucht genau ein Label.")

# Die Aufteilung trennt Lernen und faire Bewertung.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

# Alle Pipelines nutzen dieselbe TF-IDF-Vektorisierung.
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistische Regression": LogisticRegression(max_iter=300, random_state=42),
    "Lineares SVM": LinearSVC(random_state=42, max_iter=3000),
}

# Jede Pipeline wird gleich trainiert und bewertet.
model_names = []
accuracies = []
for model_name, classifier in models.items():
    pipeline = Pipeline(
        [("tfidf", TfidfVectorizer()), ("classifier", classifier)]
    )
    pipeline.fit(train_texts, train_labels)
    predictions = pipeline.predict(test_texts)
    accuracies.append(accuracy_score(test_labels, predictions))
    model_names.append(model_name)

# Kurze Ausgaben zeigen Version, Datenmenge und Ergebnis.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsbeispiele: {len(train_texts)}, Testbeispiele: {len(test_texts)}")
for model_name, score in zip(model_names, accuracies):
    print(f"{model_name}: Testgenauigkeit {score:.2f}")

# Die Balken machen den Modellvergleich schnell sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(model_names, accuracies, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Vergleich gleicher TF-IDF-Pipeline mit drei Modellen")
ax.set_xlabel("Klassifikator")
ax.set_ylabel("Testgenauigkeit")
ax.set_ylim(0, 1.05)
plt.show()



### **3.3. Eigenes Textprojekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_03.jpg?v=1787681549" width="250">



>* Kleine Textsammlung mit klarer Aufgabe wählen
>* Labels, Datenqualität und Herausforderungen prüfen

>* Pipeline verbindet Textschritte bis zur Vorhersage
>* Validierung trennen und Daten verantwortungsvoll nutzen

>* Modellfehler analysieren und gezielt verbessern
>* Pipeline dokumentiert speichern und verantwortungsvoll nutzen



In [ ]:
#@title Python-Code - Eigenes Textprojekt

# Wir trainieren eine kleine Textpipeline.
# TF-IDF wandelt Texte in Merkmale um.
# Die gespeicherte Pipeline sagt neue Labels voraus.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Diese kleine Sammlung ersetzt eine lokale Projektdatei.
texts = np.array([
    "Das Produkt funktioniert sehr gut und kam schnell an.",
    "Ich bin zufrieden und empfehle den Service weiter.",
    "Die Qualität ist hervorragend und die Bedienung einfach.",
    "Vielen Dank für die schnelle und freundliche Hilfe.",
    "Alles lief problemlos und das Ergebnis überzeugt mich.",
    "Die Lieferung war pünktlich und die Verpackung sauber.",
    "Das Gerät ist kaputt und der Support reagiert nicht.",
    "Ich habe eine Beschwerde wegen der langen Wartezeit.",
    "Die Rechnung ist falsch und niemand hilft mir.",
    "Das Paket kam beschädigt an und Teile fehlen.",
    "Die App stürzt ständig ab und ist unbrauchbar.",
    "Der Termin wurde ohne Erklärung abgesagt.",
    "Wie kann ich mein Passwort zurücksetzen?",
    "Wann wird meine Bestellung geliefert?",
    "Welche Zahlungsmethoden werden im Shop akzeptiert?",
    "Kann ich die Adresse nachträglich ändern?",
    "Wo finde ich die Anleitung für das Gerät?",
    "Gibt es eine Garantie für dieses Produkt?",
])

# Jedes Textbeispiel bekommt ein klares Ziel-Label.
labels = np.array([
    "Lob", "Lob", "Lob", "Lob", "Lob", "Lob",
    "Beschwerde", "Beschwerde", "Beschwerde", "Beschwerde", "Beschwerde",
    "Beschwerde", "Frage", "Frage", "Frage", "Frage", "Frage", "Frage",
])

# Eine einfache Prüfung schützt vor vertauschten Daten.
if len(texts) != len(labels):
    raise ValueError("Texte und Labels müssen gleich lang sein.")

# Die Aufteilung trennt Training und Validierung sauber.
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    texts, labels, test_size=0.33, random_state=42, stratify=labels
)

# Die Pipeline lernt Vokabular und Modell nur aus Trainingsdaten.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("model", LogisticRegression(max_iter=300, random_state=42)),
])

# Ein einziger Trainingsaufruf passt alle Pipeline-Schritte an.
pipeline.fit(train_texts, train_labels)

# Die Validierung nutzt ungesehene Texte aus der Aufteilung.
predicted_labels = pipeline.predict(valid_texts)
accuracy = accuracy_score(valid_labels, predicted_labels)

# Eine Variable simuliert das spätere Speichern der Pipeline.
saved_pipeline = pipeline
new_text = ["Meine Bestellung ist beschädigt angekommen."]
new_prediction = saved_pipeline.predict(new_text)[0]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Validierungsgenauigkeit: {accuracy:.2f}")
print(f"Neue Beispielvorhersage: {new_prediction}")

# Die Matrix zeigt, welche Klassen verwechselt wurden.
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    valid_labels, predicted_labels, ax=ax, colorbar=False
)

ax.set_title("Validierung der Textpipeline")
ax.set_xlabel("Vorhergesagtes Label")
ax.set_ylabel("Wahres Label")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Text als Merkmale**</font>


In this lecture, you learned to:
- Strukturieren und bereinigen kleine lokale Textsammlungen mit Labels. 
- Erzeugen Textmerkmale mit Count-, TF-IDF- und Hashing-Vektorisierung. 
- Trainieren, validieren, analysieren und speichern einfache Textklassifikationspipelines. 

In the next Module (Module 13), we will go over 'Bild- und Signalmodelle'